In [ ]:
%pip install pytesseract
%pip install opencv-python
!sudo apt install tesseract-ocr

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
import cv2
import pytesseract
from google.colab.patches import cv2_imshow
import numpy as np

def get_better_ocr_system(image_path):
    try:
        # Load the image
        img = cv2.imread(image_path)
        if img is None:
            raise FileNotFoundError(f"Error: Image not found at {image_path}")

        print("Original Image:")
        cv2_imshow(img)

        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # OPTION 1: Direct approach (simplest - usually works best)
        print("\n=== OPTION 1: Direct OCR ===")
        text_direct = pytesseract.image_to_string(gray, config='--oem 3 --psm 6')
        print(f"Direct OCR result:\n{text_direct}")

        # OPTION 2: Denoising and thresholding
        print("\n=== OPTION 2: Denoising ===")
        # Remove noise
        denoised = cv2.fastNlMeansDenoising(gray, h=30)

        # Apply threshold
        _, thresh = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # Check if text is white on black (invert if needed)
        # Count white vs black pixels in top-left corner
        if np.mean(thresh[:50, :50]) < 127:
            thresh = cv2.bitwise_not(thresh)

        print("Denoised and thresholded image:")
        cv2_imshow(thresh)

        text_denoised = pytesseract.image_to_string(thresh, config='--oem 3 --psm 6')
        print(f"Denoised OCR result:\n{text_denoised}")

        # OPTION 3: Adaptive thresholding (for varying lighting)
        print("\n=== OPTION 3: Adaptive Thresholding ===")
        adaptive = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                        cv2.THRESH_BINARY, 15, 2)
        print("Adaptive threshold image:")
        cv2_imshow(adaptive)

        text_adaptive = pytesseract.image_to_string(adaptive, config='--oem 3 --psm 6')
        print(f"Adaptive OCR result:\n{text_adaptive}")

        # OPTION 4: With dilation/erosion (for broken text)
        print("\n=== OPTION 4: Morphological Operations ===")
        kernel = np.ones((1, 1), np.uint8)
        dilated = cv2.dilate(thresh, kernel, iterations=1)
        eroded = cv2.erode(dilated, kernel, iterations=1)

        print("After dilation/erosion:")
        cv2_imshow(eroded)

        text_morph = pytesseract.image_to_string(eroded, config='--oem 3 --psm 6')
        print(f"Morphological OCR result:\n{text_morph}")

        # Return the best result (choose one with most characters)
        results = [
            ("Direct", text_direct),
            ("Denoised", text_denoised),
            ("Adaptive", text_adaptive),
            ("Morphological", text_morph)
        ]

        # Sort by text length (assuming longer is better if it makes sense)
        results.sort(key=lambda x: len(x[1].strip()), reverse=True)

        print("\n" + "="*50)
        print("SUMMARY OF ALL METHODS:")
        print("="*50)
        for method, text in results:
            print(f"\n{method}:\n{text.strip()}")

        return results[0][1]  # Return the longest result

    except Exception as e:
        print(f"An error occurred: {e}")
        import traceback
        traceback.print_exc()
        return None

# Test the function
image_file = '/content/sample1.jpg'  # Change to your actual filename
extracted_text = get_better_ocr_system(image_file)

if extracted_text:
    print("\n" + "="*50)
    print("FINAL SELECTED TEXT:")
    print("="*50)
    print(extracted_text)
else:
    print("Failed to extract any text")

An error occurred: Error: Image not found at /content/sample1.jpg
Failed to extract any text


Traceback (most recent call last):
  File "/tmp/ipython-input-3982608115.py", line 11, in get_better_ocr_system
    raise FileNotFoundError(f"Error: Image not found at {image_path}")
FileNotFoundError: Error: Image not found at /content/sample1.jpg


In [ ]:
import cv2
import pytesseract
import numpy as np

def get_better_ocr_system(image_path):
    try:
        # Load the image
        img = cv2.imread(image_path)
        if img is None:
            raise FileNotFoundError(f"Error: Image not found at {image_path}")

        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Remove noise
        denoised = cv2.fastNlMeansDenoising(gray, h=30)

        # Apply threshold
        _, thresh = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # Invert if needed (ensure black text on white background)
        if np.mean(thresh[:50, :50]) < 127:
            thresh = cv2.bitwise_not(thresh)

        # Perform OCR
        extracted_text = pytesseract.image_to_string(thresh, config='--oem 3 --psm 6')

        return extracted_text.strip()

    except Exception as e:
        print(f"An error occurred: {e}")
        return None

# Test the function
image_file = '/content/hamid-roshaan-IGVGEFQHczg-unsplash.jpg'
extracted_text = get_better_ocr_system(image_file)

if extracted_text:
    print(extracted_text)

ModuleNotFoundError: No module named 'pytesseract'